In [ ]:
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
import glob
import os

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
import lightgbm as lgb

# Set path based on your screenshot structure
DATA_PATH = "../../Dataset/CICIDS/"

In [ ]:
import glob
import os
import pandas as pd

# Adjusted Path: '../../' moves up from 'ml_notebooks/cicids' into the 'AEGIS' root, 
# then looks into 'CICIDS' 
DATA_PATH = "../../Dataset/CICIDS/" 

# Get all CSV files
all_files = glob.glob(os.path.join(DATA_PATH, "*.csv"))

# Debugging: Print how many files were found
print(f"Found {len(all_files)} files in {DATA_PATH}")

if len(all_files) == 0:
    print("❌ ERROR: No CSV files found! Check your folder structure.")
    # Fallback: try current directory just in case
    all_files = glob.glob("*.csv")
    if len(all_files) > 0:
        print(f"Using fallback: Found {len(all_files)} files in current directory.")

li = []

# Loop through the files found
for filename in all_files:
    try:
        # Read a sample to keep the memory footprint low for the AEGIS proof-of-concept [cite: 8, 147]
        df_temp = pd.read_csv(filename, index_col=None, header=0, low_memory=False)
        
        # Take a 20k sample or the whole file if it's smaller
        sample_size = min(20000, len(df_temp))
        df_temp = df_temp.sample(n=sample_size, random_state=42)
        
        li.append(df_temp)
        print(f"Successfully loaded: {os.path.basename(filename)}")
    except Exception as e:
        print(f"Could not read {filename}: {e}")

# Only concatenate if we actually found data
if li:
    df = pd.concat(li, axis=0, ignore_index=True)
    print(f"\n✅ Total Combined Shape: {df.shape}")
else:
    print("❌ Critical Failure: List 'li' is empty. Model training cannot proceed.")

Found 8 files in ../CICIDS/
Successfully loaded: Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
Successfully loaded: Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
Successfully loaded: Friday-WorkingHours-Morning.pcap_ISCX.csv
Successfully loaded: Monday-WorkingHours.pcap_ISCX.csv
Successfully loaded: Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
Successfully loaded: Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
Successfully loaded: Tuesday-WorkingHours.pcap_ISCX.csv
Successfully loaded: Wednesday-workingHours.pcap_ISCX.csv

✅ Total Combined Shape: (160000, 79)


In [6]:
# Strip whitespace from column names (common in CICIDS-2017/2018)
df.columns = df.columns.str.strip()

# Replace infinity values and drop rows with missing data
df.replace([np.inf, -np.inf], np.nan, inplace=True)
df.dropna(inplace=True)

# Drop non-numeric metadata columns that cause overfitting in network EDRs
# These are often used as 'shortcuts' by ML models rather than learning behavior
metadata_cols = ['Flow ID', 'Source IP', 'Destination IP', 'Timestamp', 'External IP']
df = df.drop(columns=[c for c in metadata_cols if c in df.columns], errors='ignore')

print(f"Cleaned Shape for Training: {df.shape}")

Cleaned Shape for Training: (159855, 79)


In [8]:
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

# 1. Clean up column names again to be safe
df.columns = df.columns.str.strip()

# 2. Encode the 'Label' column
le = LabelEncoder()
df['Label_Encoded'] = le.fit_transform(df['Label'])

# 3. Identify and remove rare classes (less than 2 members)
# This prevents the "ValueError: The least populated classes in y have only 1 member"
class_counts = df['Label_Encoded'].value_counts()
rare_classes = class_counts[class_counts < 2].index

if len(rare_classes) > 0:
    print(f"⚠️ Removing rare classes with insufficient data: {le.inverse_transform(rare_classes)}")
    df = df[~df['Label_Encoded'].isin(rare_classes)]

# 4. Final Feature and Target preparation
# We drop the original 'Label' and the new encoded one from X
X = df.drop(columns=['Label', 'Label_Encoded'], errors='ignore')
y = df['Label_Encoded']

# Store the mapping for the AEGIS Dashboard [cite: 144]
label_mapping = dict(zip(range(len(le.classes_)), le.classes_))
print("\n✅ Final Threat Classes for Training:")
for code, label in label_mapping.items():
    if code not in rare_classes:
        print(f" - {label}: {code}")

# 5. Split data: 80% Training, 20% Testing
# Stratification ensures the model learns the correct attack ratios 
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\nSplit complete. Training rows: {len(X_train)}, Testing rows: {len(X_test)}")

⚠️ Removing rare classes with insufficient data: [8]

✅ Final Threat Classes for Training:
 - 0: 0
 - 1: 1
 - 2: 2
 - 3: 3
 - 4: 4
 - 5: 5
 - 6: 6
 - 7: 7
 - 9: 9
 - 10: 10
 - 11: 11
 - 12: 12
 - 13: 13
 - 14: 14

Split complete. Training rows: 127883, Testing rows: 31971


In [10]:
import lightgbm as lgb
from sklearn.metrics import classification_report

# Initialize LightGBM optimized for AEGIS Layer 1 [cite: 20]
model = lgb.LGBMClassifier(
    n_estimators=500,        
    learning_rate=0.05,
    max_depth=10,
    num_leaves=31,
    class_weight='balanced', 
    importance_type='gain',  
    random_state=42
)

print("Starting LightGBM Training (AEGIS Model B)...")
model.fit(X_train, y_train)
print("Training Complete.")

# Evaluate Performance
y_pred = model.predict(X_test)

print("\n--- Model Performance Report ---")
# FIX: Convert all target names to strings to avoid TypeError 
current_classes = [str(le.classes_[i]) for i in sorted(y.unique())]

print(classification_report(y_test, y_pred, target_names=current_classes))

Starting LightGBM Training (AEGIS Model B)...
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.009656 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 14534
[LightGBM] [Info] Number of data points in the train set: 127883, number of used features: 68
[LightGBM] [Info] Start training from score -2.639057
[LightGBM] [Info] Start training from score -2.639057
[LightGBM] [Info] Start training from score -2.639057
[LightGBM] [Info] Start training from score -2.639057
[LightGBM] [Info] Start training from score -2.639057
[LightGBM] [Info] Start training from score -2.639057
[LightGBM] [Info] Start training from score -2.639057
[LightGBM] [Info] Start training from score -2.639057
[LightGBM] [Info] Start training from score -2.639057
[LightGBM] [Info] Start training 

c:\Users\USER\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\USER\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\USER\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(averag

In [14]:
import joblib
import numpy as np

# 1. Generate probabilities for the voting protocol
# This corresponds to the ML Detection Pipeline in the blueprint
probs = model.predict_proba(X_test)

# 2. Calculate the Final Threat Score for Model B
# AEGIS considers the 'Attack Probability' (1 - Benign Probability)
# This feeds into the 0.3 threshold for initiating a Peer Voting Request
class_names = [str(c) for c in le.classes_]
if 'BENIGN' in class_names:
    benign_idx = class_names.index('BENIGN')
    ae_threat_scores = 1 - probs[:, benign_idx]
else:
    # Fallback if BENIGN is missing: use the most confident attack probability
    ae_threat_scores = np.max(probs, axis=1)

print(f"Sample AEGIS Threat Scores: {ae_threat_scores[:5]}")

# 3. Export the production-ready package for Layer 1 Agents
# This .pkl file will be loaded by the Python daemon on each node
export_data = {
    "model": model,
    "label_encoder": le,
    "features": X.columns.tolist(),
    "version": "1.0-LGBM",
    "metadata": "Supervised Classifier (Model B) for AEGIS Distributed EDR"
}

# Save the model
joblib.dump(export_data, "../../trained_models/cicids/aegis_lgbm_cicids_model.pkl")

print("\n✅ aegis_lgbm_cicids_model.pkl saved successfully.")
print("This completes the Model B (Supervised) training pipeline.")

Sample AEGIS Threat Scores: [1.         0.99999998 0.99999987 0.99999999 0.99999994]

✅ aegis_lgbm_cicids_model.pkl saved successfully.
This completes the Model B (Supervised) training pipeline.
